# MLOps Pipeline for Hospital Readmission Classification using the UCI Diabetes Dataset

This project implements an end-to-end MLOps pipeline for predicting hospital readmission risk using the UCI Diabetes 130-US Hospitals dataset. The objective is to build a reliable machine learning classification system that identifies whether a patient is likely to be readmitted within 30 days based on historical clinical records.

The dataset contains over 100,000 patient records with 47 features, including demographic information, diagnoses, medication history, and hospital visit details. It presents real-world challenges such as missing values in multiple formats, high-cardinality categorical variables, and significant class imbalance.

The pipeline covers data ingestion, preprocessing, feature engineering, model training, and evaluation using appropriate metrics such as ROC-AUC, precision, and recall. The project is structured with an MLOps mindset to support reproducibility, scalability, and deployment readiness in a healthcare environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("diabetic_data.csv", na_values = "?")
data.sample(10)

In [ ]:
data.shape

In [ ]:
missing = data.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_percent
})

missing_df.head(10)

The dataset contains 101,766 patient encounters and 50 features. It is a real-world clinical dataset characterized by significant missing data, particularly in laboratory and specialty-related features. The level of missingness varies across features, requiring a combination of feature removal and targeted imputation strategies to ensure data quality for modeling.

Columns with extremely high levels of missing data (typically above 80%) were dropped from the dataset as part of the data preprocessing stage. These features lacked sufficient information density to support reliable statistical imputation and would likely introduce noise or bias if retained. Removing them helps improve data quality, reduces dimensionality, and ensures the machine learning model is trained on more meaningful and informative features.

In [ ]:
data.drop(columns=['weight', 'max_glu_serum', 'A1Cresult'], inplace=True)


data['medical_specialty'] = data['medical_specialty'].fillna('missing')
data['payer_code'] = data['payer_code'].fillna('missing')

threshold = 0.01

top_specialties = data['medical_specialty'].value_counts(normalize=True)
top_specialties = top_specialties[top_specialties > threshold].index
data['medical_specialty'] = data['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'other'
)

top_payers = data['payer_code'].value_counts(normalize=True)
top_payers = top_payers[top_payers > threshold].index
data['payer_code'] = data['payer_code'].apply(
    lambda x: x if x in top_payers else 'other'
)

In [ ]:
data[['medical_specialty', 'payer_code']].isnull().sum()

In [ ]:
data['medical_specialty'].value_counts().head(10)

In [ ]:
data.duplicated().sum()

In [ ]:
data['readmitted'].value_counts(normalize=True)

The dataset was checked for duplicate records and none were found, confirming data integrity at the row level.

The target variable (readmitted) shows a moderate class imbalance:
- NO: 53.9%
- >30: 34.9%
- <30: 11.2%

This indicates a 3-class classification problem with a relatively smaller minority class (<30 days readmission), which will require careful model evaluation using metrics such as F1-score and recall rather than accuracy alone.

Missing values were consistently encoded using the label 'missing' across categorical features to ensure uniform representation of incomplete data during model training.

In [ ]:
data['race'] = data['race'].fillna('missing')
for col in ['diag_1', 'diag_2', 'diag_3']:
    data[col] = data[col].fillna('missing')

In [ ]:
data['race'] = data['race'].replace({'AfricanAmerican': 'African American'})

In [ ]:
missing = data.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(data)) * 100

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_percent': missing_percent
})

missing_df.head(10)

In [ ]:
data.isnull().sum().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.info()

In [ ]:
data.select_dtypes(include='object').nunique().sort_values()

In [ ]:

np.isinf(data.select_dtypes(include=['number'])).sum()

In [ ]:
data.describe(include='all')

In [ ]:
data['readmitted'].value_counts(normalize=True)

In [ ]:
data = pd.read_csv("IDS_mapping.csv")
data.info()
data.sample(10)